# Metaseed Python Demo

This notebook demonstrates the Python API for working with MIAPPE-compliant phenotyping metadata.

## 1. Architecture: From YAML Spec to Python API

Metaseed uses a schema-driven approach where entity definitions in YAML are transformed into Pydantic models at runtime.

```
miappe_v1.1.yaml        Single YAML file with all 14 entities
        |
        v
    SpecLoader          Parses YAML into ProfileSpec/EntitySpec
   (loader.py)
        |
        v
    EntitySpec          Pydantic model describing the spec structure
   (schema.py)
        |
        v
    ModelFactory        Dynamically creates Pydantic models
   (factory.py)         using pydantic.create_model()
        |
        v
   Pydantic Model       Runtime model with validation,
                        serialization, nested entities
        |
        v
    get_model()         Public API with caching
        |
        v
    User Code           Create instances, nest entities,
                        validate, serialize
```

This approach provides:
- **Single source of truth**: One YAML file per version defines all entities
- **Type safety**: Generated Pydantic models enforce types
- **Ontology mapping**: Fields link to PPEO/MIAPPE ontology terms
- **Extensibility**: New profiles (ISA, custom) can be added as YAML files

## 2. Getting Started

In [1]:
from metaseed import get_model, validate, SpecLoader, YamlStorage, JsonStorage

## 3. The Unified YAML Specification

All 14 MIAPPE entities are defined in a single YAML file. This is the single source of truth.

In [2]:
# Display the unified YAML spec file
from pathlib import Path
from metaseed.specs import SpecLoader

loader = SpecLoader()
spec_path = loader.get_profile_path("1.1")

print(f"Spec file: {spec_path.name}")
print(f"Location: {spec_path}")
print("=" * 60)
# Show first 100 lines
lines = spec_path.read_text().split('\n')
print('\n'.join(lines))

Spec file: miappe_v1.1.yaml
Location: /Users/sdrwacker/workspace/miappe-tools/metaseed/src/metaseed/specs/miappe_v1.1.yaml
# MIAPPE v1.1 Specification
# Minimum Information About Plant Phenotyping Experiments
# Single source of truth for all entity definitions

version: "1.1"
name: MIAPPE
description: >
  MIAPPE (Minimum Information About Plant Phenotyping Experiments) is a
  reporting standard for plant phenotyping experiments. This specification
  defines 14 entities following the Plant Phenotyping Experiment Ontology (PPEO).
ontology: PPEO

# Validation rules applied across entities
validation_rules:
  - name: date_range
    description: End date must not be before start date
    applies_to: [Study, Event]
    condition: end_date >= start_date

  - name: unique_id_pattern
    description: Unique IDs must contain only alphanumeric characters, underscores, and hyphens
    applies_to: all
    field: unique_id
    pattern: "^[A-Za-z0-9_-]+$"

# Entity definitions
entities:

  Investigat

## 4. From YAML to Python Model

Now let's trace how this YAML spec becomes a Python model.

In [3]:
# Step 1: SpecLoader reads YAML files
loader = SpecLoader()

# List available entities
entities = loader.list_entities(version="1.1")
print(f"MIAPPE v1.1 defines {len(entities)} entities:")
for entity in entities:
    print(f"  - {entity}")

MIAPPE v1.1 defines 14 entities:
  - BiologicalMaterial
  - DataFile
  - Environment
  - Event
  - Factor
  - FactorValue
  - Investigation
  - Location
  - MaterialSource
  - ObservationUnit
  - ObservedVariable
  - Person
  - Sample
  - Study


In [4]:
# Step 2: Load YAML into an EntitySpec object
spec = loader.load_entity("investigation", version="1.1")

print(f"Entity: {spec.name}")
print(f"Version: {spec.version}")
print(f"Ontology: {spec.ontology_term}")
print(f"\nFields from spec:")
for field in spec.fields:
    req = "required" if field.required else "optional"
    onto = f" [{field.ontology_term}]" if field.ontology_term else ""
    print(f"  {field.name}: {field.type.value} ({req}){onto}")

Entity: Investigation
Version: 1.1
Ontology: PPEO:investigation

Fields from spec:
  unique_id: string (required) [PPEO:hasIdentifier]
  title: string (required) [PPEO:hasName]
  description: string (optional) [PPEO:hasDescription]
  submission_date: date (optional) [PPEO:hasSubmissionDate]
  public_release_date: date (optional) [PPEO:hasPublicReleaseDate]
  license: uri (optional) [PPEO:hasLicense]
  miappe_version: string (optional) [PPEO:hasMIAPPEVersion]
  associated_publications: list (optional) [PPEO:hasAssociatedPublication]
  contacts: list (optional) [PPEO:hasPersonWithRole]
  studies: list (optional) [PPEO:hasPart]


In [5]:
# Step 3: get_model() creates a Pydantic model from the spec
Investigation = get_model("Investigation", version="1.1")

print(f"Generated model: {Investigation.__name__}")
print(f"Base class: {Investigation.__bases__}")
print(f"\nModel fields (from Pydantic):")
for name, field in Investigation.model_fields.items():
    print(f"  {name}: {field.annotation}")

Generated model: Investigation
Base class: (<class 'metaseed.models.factory.MIAPPEBaseModel'>,)

Model fields (from Pydantic):
  unique_id: <class 'str'>
  title: <class 'str'>
  description: typing.Optional[typing.Annotated[str, FieldInfo(annotation=NoneType, required=True, description='Detailed description of the investigation.')]]
  submission_date: typing.Optional[typing.Annotated[datetime.date, FieldInfo(annotation=NoneType, required=True, description='Date submitted to archive. Format: ISO 8601 (YYYY-MM-DD).')]]
  public_release_date: typing.Optional[typing.Annotated[datetime.date, FieldInfo(annotation=NoneType, required=True, description='Date of public release. Format: ISO 8601 (YYYY-MM-DD).')]]
  license: typing.Optional[typing.Annotated[pydantic.networks.AnyUrl, FieldInfo(annotation=NoneType, required=True, description='URL to the license document.')]]
  miappe_version: typing.Optional[typing.Annotated[str, FieldInfo(annotation=NoneType, required=True, description='Version of

## 5. PPEO Entity Hierarchy

MIAPPE entities follow the Plant Phenotyping Experiment Ontology (PPEO) hierarchy:

```
Investigation
  |-- contacts: [Person]
  |-- studies: [Study]
        |-- persons: [Person]
        |-- biological_materials: [BiologicalMaterial]
        |-- observation_units: [ObservationUnit]
        |-- observed_variables: [ObservedVariable]
        |-- factors: [Factor]
        |     |-- values: [FactorValue]
        |-- events: [Event]
        |-- environments: [Environment]
        |-- data_files: [DataFile]
```

Entities can be nested directly in Python using standard list operations.

## 6. Building a Complete Investigation

Let's build a complete investigation with nested entities.

In [6]:
import datetime

# Get all models we need
Investigation = get_model("Investigation")
Study = get_model("Study")
Person = get_model("Person")
BiologicalMaterial = get_model("BiologicalMaterial")
ObservedVariable = get_model("ObservedVariable")
Factor = get_model("Factor")

In [7]:
# Build the investigation with nested entities in constructor
investigation = Investigation(
    unique_id="INV-DROUGHT-2024",
    title="Multi-Environment Drought Tolerance Assessment in European Wheat",
    description="Evaluating drought tolerance mechanisms across 50 European wheat "
                "cultivars under controlled water stress conditions.",
    submission_date=datetime.date(2024, 6, 1),
    public_release_date=datetime.date(2025, 1, 1),
    license="https://creativecommons.org/licenses/by/4.0/",
    miappe_version="1.1",
    
    # Nested contacts
    contacts=[
        Person(
            name="Dr. Jane Smith",
            email="j.smith@university.edu",
            institution="University Research Center",
            role="Principal Investigator",
            orcid="0000-0001-2345-6789",
        ),
        Person(
            name="Dr. John Doe",
            email="j.doe@university.edu",
            institution="University Research Center",
            role="Data Manager",
        ),
    ],
    
    # Nested studies
    studies=[
        Study(
            unique_id="STU-2024-SITE-A",
            title="Field Trial 2024 - Site A (Germany)",
            description="First year field trial at research station A",
            start_date=datetime.date(2024, 4, 1),
            end_date=datetime.date(2024, 9, 30),
            geographic_location="Germany",
            latitude=52.52,
            longitude=13.405,
            experimental_design_type="Randomized Complete Block Design",
            growth_facility_type="field",
            
            # Nested biological materials
            biological_materials=[
                BiologicalMaterial(
                    unique_id="BM-001",
                    organism="Triticum aestivum",
                    genus="Triticum",
                    species="aestivum",
                    infraspecific_name="cv. Apache",
                ),
                BiologicalMaterial(
                    unique_id="BM-002",
                    organism="Triticum aestivum",
                    genus="Triticum",
                    species="aestivum",
                    infraspecific_name="cv. Soissons",
                ),
            ],
            
            # Nested observed variables
            observed_variables=[
                ObservedVariable(
                    unique_id="VAR-001",
                    name="Plant Height",
                    trait="Plant height",
                    trait_accession_number="TO:0000207",
                    method="Ruler measurement",
                    scale="cm",
                ),
                ObservedVariable(
                    unique_id="VAR-002",
                    name="Leaf Wilting Score",
                    trait="Leaf wilting",
                    trait_accession_number="TO:0000109",
                    method="Visual scoring 1-9",
                    scale="ordinal",
                ),
            ],
            
            # Nested factors
            factors=[
                Factor(
                    unique_id="FACTOR-001",
                    name="Water Treatment",
                    description="Water stress treatment levels",
                ),
            ],
        ),
    ],
)

In [8]:
# Explore the nested structure
print(f"Investigation: {investigation.title}")
print(f"  Contacts: {len(investigation.contacts)}")
print(f"  Studies: {len(investigation.studies)}")

study = investigation.studies[0]
print(f"\n  Study: {study.title}")
print(f"    Biological Materials: {len(study.biological_materials)}")
print(f"    Observed Variables: {len(study.observed_variables)}")
print(f"    Factors: {len(study.factors)}")

Investigation: Multi-Environment Drought Tolerance Assessment in European Wheat
  Contacts: 2
  Studies: 1

  Study: Field Trial 2024 - Site A (Germany)
    Biological Materials: 2
    Observed Variables: 2
    Factors: 1


In [9]:
# You can also build incrementally using standard list operations
investigation.studies[0].biological_materials.append(
    BiologicalMaterial(
        unique_id="BM-003",
        organism="Triticum aestivum",
        genus="Triticum",
        species="aestivum",
        infraspecific_name="cv. Bologna",
    )
)

print(f"Biological Materials after append: {len(investigation.studies[0].biological_materials)}")

Biological Materials after append: 3


## 7. Cascading Validation

The `validate()` function can validate an entire entity tree, checking all nested entities.

In [10]:
# Validate the complete investigation (cascading to nested entities)
errors = validate(investigation, cascade=True)

if errors:
    print("Validation errors:")
    for error in errors:
        print(f"  {error.field}: {error.message}")
else:
    print("Validation passed for investigation and all nested entities!")

Validation passed for investigation and all nested entities!


In [11]:
# Demonstrate validation error with invalid date range in nested study
invalid_inv = Investigation(
    unique_id="INV-BAD",
    title="Invalid Investigation",
    studies=[
        Study(
            unique_id="STU-001",
            title="Study with bad dates",
            start_date=datetime.date(2024, 12, 31),
            end_date=datetime.date(2024, 1, 1),  # End before start!
        ),
    ],
)

errors = validate(invalid_inv, cascade=True)
print("Cascading validation errors:")
for error in errors:
    print(f"  {error.field}: {error.message}")

Cascading validation errors:
  studies[0].end_date: end_date (2024-01-01) must not be before start_date (2024-12-31)


## 8. Serialization to YAML

Nested entities serialize to YAML preserving the full structure.

In [12]:
# Save to YAML file
import tempfile

temp_dir = Path(tempfile.mkdtemp())
yaml_storage = YamlStorage()

output_path = temp_dir / "investigation.yaml"
yaml_storage.save(investigation, output_path)

print(f"Saved to: {output_path}")
print("\n" + "=" * 60)
print("YAML OUTPUT:")
print("=" * 60)
print(output_path.read_text())

Saved to: /var/folders/hr/bq19zbjx0wvbr5gmvwypgb7431fw57/T/tmpkwrdtpda/investigation.yaml

YAML OUTPUT:
unique_id: INV-DROUGHT-2024
title: Multi-Environment Drought Tolerance Assessment in European Wheat
description: Evaluating drought tolerance mechanisms across 50 European wheat cultivars
  under controlled water stress conditions.
submission_date: '2024-06-01'
public_release_date: '2025-01-01'
license: https://creativecommons.org/licenses/by/4.0/
miappe_version: '1.1'
associated_publications: []
contacts:
- name: Dr. Jane Smith
  email: j.smith@university.edu
  institution: University Research Center
  role: Principal Investigator
  orcid: 0000-0001-2345-6789
- name: Dr. John Doe
  email: j.doe@university.edu
  institution: University Research Center
  role: Data Manager
studies:
- unique_id: STU-2024-SITE-A
  title: Field Trial 2024 - Site A (Germany)
  description: First year field trial at research station A
  start_date: '2024-04-01'
  end_date: '2024-09-30'
  geographic_locatio

## 9. Loading from Files

In [13]:
# Load from YAML back into Python models
loaded_inv = yaml_storage.load(output_path, Investigation)

print(f"Loaded: {loaded_inv.title}")
print(f"Studies: {len(loaded_inv.studies)}")
print(f"First study: {loaded_inv.studies[0].title}")
print(f"Biological materials: {len(loaded_inv.studies[0].biological_materials)}")

# Verify data integrity
print(f"\nMaterials:")
for mat in loaded_inv.studies[0].biological_materials:
    print(f"  {mat.unique_id}: {mat.organism} {mat.infraspecific_name}")

Loaded: Multi-Environment Drought Tolerance Assessment in European Wheat
Studies: 1
First study: Field Trial 2024 - Site A (Germany)
Biological materials: 3

Materials:
  BM-001: Triticum aestivum cv. Apache
  BM-002: Triticum aestivum cv. Soissons
  BM-003: Triticum aestivum cv. Bologna


## 10. JSON Schema Export

In [14]:
import json

schema = Investigation.model_json_schema()

print("Investigation JSON Schema:")
print(json.dumps(schema, indent=2))

Investigation JSON Schema:
{
  "additionalProperties": false,
  "properties": {
    "unique_id": {
      "description": "Unique identifier for the investigation.",
      "maxLength": 100,
      "minLength": 1,
      "pattern": "^[A-Za-z0-9_-]+$",
      "title": "Unique Id",
      "type": "string"
    },
    "title": {
      "description": "Human-readable title of the investigation.",
      "maxLength": 255,
      "minLength": 1,
      "title": "Title",
      "type": "string"
    },
    "description": {
      "anyOf": [
        {
          "description": "Detailed description of the investigation.",
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Description"
    },
    "submission_date": {
      "anyOf": [
        {
          "description": "Date submitted to archive. Format: ISO 8601 (YYYY-MM-DD).",
          "format": "date",
          "type": "string"
        },
        {
          "type": "null"
    

## Summary

This notebook demonstrated the complete flow:

1. **Unified YAML Spec**: All entities in `miappe_v1.1.yaml`
2. **SpecLoader**: Parses YAML into `ProfileSpec` and `EntitySpec` objects
3. **ModelFactory**: Creates Pydantic models with `pydantic.create_model()`
4. **get_model()**: Public API with caching and name normalization
5. **Nested Entities**: Follow PPEO hierarchy (Investigation -> Study -> ...)
6. **Cascading Validation**: `validate(entity, cascade=True)` checks entire tree
7. **Serialization**: Full structure preserved in YAML/JSON

For more information, see the [documentation](https://metaseed.readthedocs.io).